# Run Three-Version Benchmark

**Purpose:**
- Use the T112A shared runner to run selected dataset rows through baseline, dflash, and cc-dflash.
- Safely manage configuration so execution doesn't unexpectedly load models without permission.

In [ ]:
import sys
from pathlib import Path

_cwd = Path.cwd()
if _cwd.name != "notebooks":
    sys.path.insert(0, str(_cwd / "notebooks"))

from utils import setup_root
ROOT = setup_root()

## Configuration

In [ ]:
DATASET = "gsm8k"
LIMIT = 1
SEED = 42
MAX_NEW_TOKENS = 128
CONDITIONS = [
    "baseline_ar",
    "dflash_r1",
    "cc_dflash_r2",
]

## 1. Selected Input

In [ ]:
import json
from datetime import datetime, timezone

filename = "gsm8k_100.jsonl" if DATASET == "gsm8k" else "qmsum_meeting_qa_100.jsonl"
eval_path = ROOT / "data/eval" / filename
if not eval_path.exists():
    raise FileNotFoundError(f"Missing data: {eval_path}")

with open(eval_path, "r", encoding="utf-8") as f:
    rows = [json.loads(line) for line in f if line.strip()]
rows = rows[:LIMIT]

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

In [ ]:
row = rows[0]
print(f"Fixture ID: {row.get('id')}")
print(f"Question: {row.get('question')}")
if 'expected_answer' in row:
    print(f"Expected Answer: {row.get('expected_answer')}")

In [ ]:
import os
from ccdf.demo import DemoRunner
from ccdf.config import load_config
from ccdf.demo.adapters.gsm8k import gsm8k_row_to_request
from ccdf.demo.adapters.qmsum import qmsum_row_to_request

try:
    config = load_config("config.yml")
except FileNotFoundError:
    config = {}

if isinstance(config, dict):
    config["dry_run"] = os.environ.get("CCDF_NOTEBOOK_TEST_MODE") == "1"
else:
    config.dry_run = os.environ.get("CCDF_NOTEBOOK_TEST_MODE") == "1"

runner = DemoRunner(config)
results_by_condition = {}

def request_for_condition(cond):
    row = rows[0]
    if DATASET == "gsm8k":
        return gsm8k_row_to_request(row, cond, seed=SEED, max_new_tokens=MAX_NEW_TOKENS)
    else:
        return qmsum_row_to_request(row, cond, seed=SEED, max_new_tokens=MAX_NEW_TOKENS)

## 2. Baseline-AR

In [ ]:
req = request_for_condition("baseline_ar")
print("Running Baseline-AR...")
baseline_result = runner.run(req)
results_by_condition["baseline_ar"] = baseline_result
print("Baseline-AR complete.")

In [ ]:
res = results_by_condition["baseline_ar"]
req_dict = res["request"]
resp = res["response"]
toks = res["tokens"]
timing = res["timing_ms"]
through = res["throughput"]
resrc = res["resources"]
qual = res["quality"]
stat = res["status"]

print("Condition name: Baseline-AR")
print("\nGenerated output:")
print(resp["generated_text"])
print(f"\nOriginal input tokens: {toks['original_input_tokens']}")
print("Compressed input tokens: —")
print("Compression ratio: —")
print(f"Output tokens: {resp['output_tokens']}")
print("Compression latency: 0 ms")
print(f"Prefill latency: {timing['prefill']:.2f} ms" if timing['prefill'] is not None else "Prefill latency: N/A")
print(f"Generation latency: {timing['generation']:.2f} ms")
print(f"End-to-end latency: {timing['e2e']:.2f} ms")
print(f"Generation tok/s: {through['generation_tok_s']:.2f}" if through['generation_tok_s'] is not None else "Generation tok/s: N/A")
print(f"End-to-end tok/s: {through['e2e_tok_s']:.2f}" if through['e2e_tok_s'] is not None else "End-to-end tok/s: N/A")
print(f"Peak VRAM: {resrc['peak_vram_gib']:.2f} GiB" if resrc['peak_vram_gib'] is not None else "Peak VRAM: N/A")
print(f"Finish reason: {resp['finish_reason']}")
print(f"Quality/evaluation status: {qual['evaluation_status']}")
if qual['evaluation_status'] != "not_evaluated":
    print(f"Quality (Match): {qual['numeric_match']}")
if not stat['ok']:
    print(f"Error details: {stat['error_type']} - {stat['error_message']}")

## 3. DFlash-R1

In [ ]:
req = request_for_condition("dflash_r1")
print("Running DFlash-R1...")
dflash_result = runner.run(req)
results_by_condition["dflash_r1"] = dflash_result
print("DFlash-R1 complete.")

In [ ]:
res = results_by_condition["dflash_r1"]
req_dict = res["request"]
resp = res["response"]
toks = res["tokens"]
timing = res["timing_ms"]
through = res["throughput"]
resrc = res["resources"]
qual = res["quality"]
stat = res["status"]

print("Condition name: DFlash-R1")
print("\nGenerated output:")
print(resp["generated_text"])
print(f"\nOriginal input tokens: {toks['original_input_tokens']}")
print("Compressed input tokens: —")
print("Compression ratio: —")
print(f"Output tokens: {resp['output_tokens']}")
print("Compression latency: 0 ms")
print(f"Prefill latency: {timing['prefill']:.2f} ms" if timing['prefill'] is not None else "Prefill latency: N/A")
print(f"Generation latency: {timing['generation']:.2f} ms")
print(f"End-to-end latency: {timing['e2e']:.2f} ms")
print(f"Generation tok/s: {through['generation_tok_s']:.2f}" if through['generation_tok_s'] is not None else "Generation tok/s: N/A")
print(f"End-to-end tok/s: {through['e2e_tok_s']:.2f}" if through['e2e_tok_s'] is not None else "End-to-end tok/s: N/A")
print(f"Peak VRAM: {resrc['peak_vram_gib']:.2f} GiB" if resrc['peak_vram_gib'] is not None else "Peak VRAM: N/A")
print(f"Finish reason: {resp['finish_reason']}")
print(f"Quality/evaluation status: {qual['evaluation_status']}")
if qual['evaluation_status'] != "not_evaluated":
    print(f"Quality (Match): {qual['numeric_match']}")
if not stat['ok']:
    print(f"Error details: {stat['error_type']} - {stat['error_message']}")

## 4. CC-DFlash-R2 Light GPU

In [ ]:
req = request_for_condition("cc_dflash_r2")
print("Running CC-DFlash-R2 Light GPU...")
cc_dflash_result = runner.run(req)
results_by_condition["cc_dflash_r2"] = cc_dflash_result
print("CC-DFlash-R2 complete.")

In [ ]:
res = results_by_condition["cc_dflash_r2"]
req_dict = res["request"]
resp = res["response"]
toks = res["tokens"]
timing = res["timing_ms"]
through = res["throughput"]
resrc = res["resources"]
qual = res["quality"]
stat = res["status"]

print("Condition name: CC-DFlash-R2 Light GPU")
print("\nGenerated output:")
print(resp["generated_text"])
print(f"\nOriginal input tokens: {toks['original_input_tokens']}")
print(f"Compressed input tokens: {toks['compressed_input_tokens']}")
print(f"Compression ratio: {toks['compression_ratio']:.2f}" if toks['compression_ratio'] is not None else "Compression ratio: N/A")
print(f"Output tokens: {resp['output_tokens']}")
print(f"Compression latency: {timing['compression']:.2f} ms" if timing['compression'] is not None else "Compression latency: N/A")
print(f"Prefill latency: {timing['prefill']:.2f} ms" if timing['prefill'] is not None else "Prefill latency: N/A")
print(f"Generation latency: {timing['generation']:.2f} ms")
print(f"End-to-end latency: {timing['e2e']:.2f} ms")
print(f"Generation tok/s: {through['generation_tok_s']:.2f}" if through['generation_tok_s'] is not None else "Generation tok/s: N/A")
print(f"End-to-end tok/s: {through['e2e_tok_s']:.2f}" if through['e2e_tok_s'] is not None else "End-to-end tok/s: N/A")
print(f"Peak VRAM: {resrc['peak_vram_gib']:.2f} GiB" if resrc['peak_vram_gib'] is not None else "Peak VRAM: N/A")
print(f"Finish reason: {resp['finish_reason']}")
print(f"Quality/evaluation status: {qual['evaluation_status']}")
if qual['evaluation_status'] != "not_evaluated":
    print(f"Quality (Match): {qual['numeric_match']}")
if not stat['ok']:
    print(f"Error details: {stat['error_type']} - {stat['error_message']}")

## 5. Three-Version Comparison

In [ ]:
import pandas as pd

comparison_rows = []
for cond in CONDITIONS:
    res = results_by_condition.get(cond)
    if res is None:
        continue
    req_dict = res["request"]
    resp = res["response"]
    toks = res["tokens"]
    timing = res["timing_ms"]
    through = res["throughput"]
    resrc = res["resources"]
    qual = res["quality"]
    stat = res["status"]
    
    comparison_rows.append({
        "Condition": "Baseline-AR" if cond == "baseline_ar" else ("DFlash-R1" if cond == "dflash_r1" else "CC-DFlash-R2 Light GPU"),
        "Input tokens": toks["original_input_tokens"],
        "Compressed tokens": toks["compressed_input_tokens"] if cond == "cc_dflash_r2" and toks["compressed_input_tokens"] is not None else "—",
        "Compression ratio": f"{toks['compression_ratio']:.2f}" if cond == "cc_dflash_r2" and toks["compression_ratio"] is not None else "—",
        "Output tokens": resp["output_tokens"],
        "T_compress (ms)": f"{timing['compression']:.2f}" if cond == "cc_dflash_r2" and timing['compression'] is not None else "0",
        "T_prefill (ms)": f"{timing['prefill']:.2f}" if timing['prefill'] is not None else "N/A",
        "T_generation (ms)": f"{timing['generation']:.2f}",
        "T_e2e (ms)": f"{timing['e2e']:.2f}",
        "Generation tok/s": f"{through['generation_tok_s']:.2f}" if through['generation_tok_s'] is not None else "N/A",
        "E2E tok/s": f"{through['e2e_tok_s']:.2f}" if through['e2e_tok_s'] is not None else "N/A",
        "Peak VRAM (GiB)": f"{resrc['peak_vram_gib']:.2f}" if resrc['peak_vram_gib'] is not None else "N/A",
        "Status": "OK" if stat["ok"] else "ERROR"
    })

df_compare = pd.DataFrame(comparison_rows)
display(df_compare)

## 6. Saved Results

In [ ]:
import json
import pandas as pd
from ccdf.demo.writers import write_jsonl_append, write_json

run_dir = ROOT / "results/charts/notebook_demo" / RUN_ID
run_dir.mkdir(parents=True, exist_ok=True)

out_jsonl = run_dir / "results.jsonl"
out_csv = run_dir / "comparison.csv"
summary_path = run_dir / "summary.json"
manifest_path = run_dir / "manifest.json"

# Write results.jsonl
for cond in CONDITIONS:
    res = results_by_condition.get(cond)
    if res is not None:
        write_jsonl_append(res, out_jsonl)

# Write comparison.csv
csv_rows = []
for cond in CONDITIONS:
    res = results_by_condition.get(cond)
    if res is None:
        continue
    req_dict = res["request"]
    resp = res["response"]
    toks = res["tokens"]
    timing = res["timing_ms"]
    through = res["throughput"]
    resrc = res["resources"]
    qual = res["quality"]
    stat = res["status"]
    
    csv_rows.append({
        "condition": req_dict["condition"],
        "display_name": "Baseline-AR" if cond == "baseline_ar" else ("DFlash-R1" if cond == "dflash_r1" else "CC-DFlash-R2 Light GPU"),
        "input_tokens": toks["original_input_tokens"],
        "compressed_tokens": toks["compressed_input_tokens"] if cond == "cc_dflash_r2" and toks["compressed_input_tokens"] is not None else "",
        "compression_ratio": toks["compression_ratio"] if cond == "cc_dflash_r2" and toks["compression_ratio"] is not None else "",
        "output_tokens": resp["output_tokens"],
        "t_compress_ms": timing["compression"] if cond == "cc_dflash_r2" and timing["compression"] is not None else 0.0,
        "t_prefill_ms": timing["prefill"] if timing["prefill"] is not None else "",
        "t_generation_ms": timing["generation"],
        "t_e2e_ms": timing["e2e"],
        "generation_tok_s": through["generation_tok_s"] if through["generation_tok_s"] is not None else "",
        "e2e_tok_s": through["e2e_tok_s"] if through["e2e_tok_s"] is not None else "",
        "peak_vram_gib": resrc["peak_vram_gib"] if resrc["peak_vram_gib"] is not None else "",
        "quality_status": qual["evaluation_status"],
        "run_status": "OK" if stat["ok"] else "ERROR"
    })
df_csv = pd.DataFrame(csv_rows)
df_csv.to_csv(out_csv, index=False)

# Write summary.json
best_throughput = ""
lowest_latency = ""
if csv_rows:
    valid_rows = [r for r in csv_rows if r["generation_tok_s"] != ""]
    if valid_rows:
        best_throughput = max(valid_rows, key=lambda x: x["generation_tok_s"])["display_name"]
    valid_e2e = [r for r in csv_rows if r["t_e2e_ms"] != ""]
    if valid_e2e:
        lowest_latency = min(valid_e2e, key=lambda x: x["t_e2e_ms"])["display_name"]

summary_data = {
    "run_id": RUN_ID,
    "dataset": DATASET,
    "sample_count": LIMIT,
    "conditions": ["Baseline-AR", "DFlash-R1", "CC-DFlash-R2 Light GPU"],
    "best_generation_throughput": best_throughput,
    "lowest_e2e_latency": lowest_latency,
    "compression_observed": any(row["compressed_tokens"] != "" for row in csv_rows),
    "claim_note": "This demo run is not a full benchmark conclusion."
}
write_json(summary_data, summary_path, overwrite=True)

# Write manifest.json
manifest_data = {
    "run_id": RUN_ID,
    "dataset": DATASET,
    "timestamp": RUN_ID,
    "schema_version": "cc_dflash_demo_v1",
    "execution_settings": {
        "limit": LIMIT,
        "seed": SEED,
        "max_new_tokens": MAX_NEW_TOKENS
    },
    "paths": {
        "results_jsonl": str(out_jsonl.relative_to(ROOT)),
        "comparison_csv": str(out_csv.relative_to(ROOT)),
        "summary_json": str(summary_path.relative_to(ROOT))
    }
}
write_json(manifest_data, manifest_path, overwrite=True)

# Write latest_run.json
latest_run_data = {
    "run_id": RUN_ID,
    "dataset": DATASET,
    "run_dir": f"results/charts/notebook_demo/{RUN_ID}",
    "table_dir": f"results/charts/notebook_demo/{RUN_ID}",
    "summary_dir": f"results/charts/notebook_demo/{RUN_ID}",
    "figure_dir": f"results/charts/notebook_demo/{RUN_ID}/charts",
    "completed": True
}
write_json(latest_run_data, ROOT / "results/charts/notebook_demo/latest_run.json", overwrite=True)

## Final Paths and Interpretation Notes

- Artifacts saved:
  - Results: `results/charts/notebook_demo/{{RUN_ID}}/results.jsonl`
  - Comparison: `results/charts/notebook_demo/{{RUN_ID}}/comparison.csv`
  - Summary: `results/charts/notebook_demo/{{RUN_ID}}/summary.json`
  - Manifest: `results/charts/notebook_demo/{{RUN_ID}}/manifest.json`
  - Latest Run: `results/charts/notebook_demo/latest_run.json`

**Semantic Correctness Notice:**
- For QMSum, **Semantic correctness is not claimed; quality fields are bounded proxies only.**